<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<span style="display:inline-block;margin-bottom:10px;padding:2px 12px;border-radius:999px;background:#6366f1;color:#fff;font:600 11px/20px system-ui,sans-serif;letter-spacing:.05em;text-transform:uppercase">Course</span>
<h1 style="font-size:1.35em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Build a production voice front desk</h1>
<p style="margin:.55em 0">This course rebuilds, step by step, the backend in <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">app/</code>: a voice agent on OpenAI Realtime that answers a
clinic's phone: it verifies callers with a texted code, registers new patients, finds free slots, books and cancels.</p>
<p style="margin:.55em 0">The design goal: <b>adding a capability is writing one typed Python function</b>. Everything else (schemas, validation,
guards, prompts, transport, recovery) is built once, here.</p>
<div style="display:flex;flex-direction:column;align-items:flex-start;gap:4px;margin:.8em 0">
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>1 settings</b>: one TOML file + environment overrides, strict</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>2 skill registry</b>: functions → tool schemas, validation, guards, timeouts</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>3 storage + rules</b>: SQLite, patient and appointment rules</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>4 skills</b>: identity, registration, appointments, clinic info</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>5 session</b>: one conversation's state, the agent's prompt</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>6 realtime client</b>: the audio + tool-call loop</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>7 resilience</b>: surviving dead connections</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>8 server</b>: browser ↔ OpenAI relay, limits, health, metrics</div>
  <div style="border:1px solid rgba(127,127,127,.35);background:rgba(127,127,127,.05);border-radius:8px;padding:8px 12px"><b>9 production</b>: config, container, the self-check</div>
</div>
<p style="margin:.55em 0">Every step writes real files into <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">course/build/</code> with <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">%%writefile</code> and then tests them. Run the
notebooks in order. All tests are free: a local fake OpenAI stands in for the real one.</p>

</div>

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">Setup</h2>
<p style="margin:.55em 0">Python 3.11+ (for <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">tomllib</code>) and one package, <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">websockets</code>. Start from an empty <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">build/</code>:</p>

</div>

In [1]:
import shutil, sys
from pathlib import Path

assert sys.version_info >= (3, 11), sys.version
import websockets

BUILD = Path('build')
if BUILD.exists():
    shutil.rmtree(BUILD)
for d in ('app/skills', 'app/domain', 'app/static', 'config', 'prompts', 'assets', 'data'):
    (BUILD / d).mkdir(parents=True)
sys.version.split()[0], websockets.__version__, sorted(str(p) for p in BUILD.iterdir())

('3.11.0',
 '17.4',
 ['build\\app',
  'build\\assets',
  'build\\config',
  'build\\data',
  'build\\prompts'])

<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<h2 style="font-size:1.2em;margin:.3em 0 .5em;line-height:1.3;font-weight:700;border:0;padding:0">The package</h2>
<p style="margin:.55em 0">A one-line package marker with a version.</p>

</div>

In [2]:
%%writefile build/app/__init__.py
"""Voice front desk backend: OpenAI Realtime + skills discovered from app/skills/. Run: python -m app"""
__version__ = "1.0.0"


Writing build/app/__init__.py


<div style="max-width:860px;border:1px solid rgba(127,127,127,.35);border-radius:10px;padding:18px 20px;margin:4px 0;background:transparent;font-family:var(--vscode-editor-font-family,var(--jp-code-font-family,Consolas,monospace));font-size:var(--vscode-editor-font-size,var(--jp-code-font-size,14px));line-height:1.6">

<p style="margin:.55em 0"><b>Next:</b> <code style="font:inherit;background:rgba(127,127,127,.15);color:#e83e8c;padding:.1em .35em;border-radius:4px">01_settings.ipynb</code>: one place for every tunable.</p>

</div>